# Decode posterior

## Load the model

In [1]:
import torch
from torch.utils.data import DataLoader

from modules.model import VariationalAutoencoder
from modules.dataset import LogMinMaxScale, EnsembleDataset
from modules.generation import reconstruct_vae_samples, save_samples_with_mean

checkpoint = torch.load('output/vae07.pt', map_location='cpu')

## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
scale     = checkpoint['SCALE']
transform = LogMinMaxScale(min_value, max_value, scale)

# Model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'], in_shape=checkpoint['IN_SHAPE'])
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [2]:
device = torch.device('cpu')
latent_dim = checkpoint['LATENT_DIM']

## Load the train dataset (only coordinates are required)

In [24]:
import xarray as xr
import numpy as np

fname = checkpoint['FNAME_TRAIN']
varkey = checkpoint['VARKEY']

## Loading raw data
ds = xr.open_dataset(fname)

lat = ds["lat"].values
lon = ds["lon"].values

## Load the posterior ensemble (latent space)

In [17]:
fname = "output/posterior_latent_vae07.nc"

ds = xr.open_dataset(fname)

N_ens = ds.sizes['ens']

Z_analysis = ds['z'].to_numpy()
Z_mean_analysis = ds['z_mean'].to_numpy()

In [15]:
latent_dim, N_ens

(32, 1000)

## Decode the posterior

In [20]:
model.eval()

decoded = []
with torch.no_grad():
    z = torch.from_numpy(Z_analysis).float().to(device)
    x_generated = model.decode(z)
    x_generated_raw = transform.invert(x_generated).squeeze()

In [21]:
X_analysis = x_generated_raw.cpu().numpy()
X_mean = X_analysis.mean(axis=0)

In [26]:
posterior_samples = xr.Dataset(
    data_vars={
        "samples": (
            ("ens", "lat", "lon"),
            X_analysis.astype(np.float32)
        ),
        "mean": (
            ("lat", "lon"),
            X_mean.astype(np.float32)
        ),
    },
    coords={
        "ens": np.arange(N_ens),
        "lat": lat,
        "lon": lon,
    },
)

posterior_samples.to_netcdf("output/posterior_samples_vae07.nc")